In [5]:
# Importing all required libraries
import re
import nltk
import string
import pickle
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Download stopwords
nltk.download('stopwords')

# Reading the dataset file
df = pd.read_csv("dataset.csv", on_bad_lines='skip', engine='python')

# Quick look at the data
print("\nFirst few rows of the dataset:")
print(df.head())

# Basic information about dataset
print("\nDataset info:")
print(df.info())

# Checking how many positive and negative samples we have
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

# Text Preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # Handle unexpected values
    if not isinstance(text, str):
        return ""

    # Convert everything to lowercase
    text = text.lower()

    # Remove links
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation marks
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Keep only alphabets
    text = re.sub(r'[^a-z\s]', '', text)

    # Split sentence into words
    tokens = text.split()

    # Remove stopwords and apply stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]

    # Join words back into sentence
    return " ".join(tokens)

# Apply preprocessing to all reviews
df['clean_text'] = df['review'].apply(preprocess_text)
print("\nSample cleaned text:")
print(df[['review', 'clean_text']].head())

# Feature Engineering

# Bag of Words representation
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

# TF-IDF representation
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

# Converting labels to numbers
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0,
    'neutral': 2
})
y = df['sentiment']

# Splitting data
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

# Training Models

# Logistic Regression (usually performs best for text data)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Naive Bayes (fast and works well with text)
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Decision Tree (not ideal for text but used for comparison)
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Evaluating Models
def evaluate(y_test, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    print("\nDetailed Report:\n", classification_report(y_test, y_pred))

# Running evaluation for all models
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")

# Comparing Results
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_nb, average='weighted'),
        f1_score(y_test, y_pred_dt, average='weighted')
    ]
})
print("\nModel Comparison:")
print(results)

# Saving trained model and vectorizer for future use
pickle.dump(lr, open("sentiment_model.pkl", "wb"))
pickle.dump(tfidf, open("vectorizer.pkl", "wb"))
print("\nModel saved successfully!")

# Testing with custom input
def predict_sentiment(text):
    cleaned = preprocess_text(text)
    vector = tfidf.transform([cleaned])
    prediction = lr.predict(vector)
    if prediction[0] == 1:
        return "Positive"
    elif prediction[0] == 0:
        return "Negative"
    else:
        return "Neutral"

# Example prediction
print("\nSample Prediction:")
print(predict_sentiment("This product is amazing and worth buying!"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



First few rows of the dataset:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7088 entries, 0 to 7087
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     7088 non-null   object
 1   sentiment  7088 non-null   object
dtypes: object(2)
memory usage: 110.9+ KB
None

Sentiment distribution:
sentiment
negative    3547
positive    3541
Name: count, dtype: int64

Sample cleaned text:
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br